# GBPUSD vs EURUSD — January 2025 mini-run

Minimal pipeline: load Jan 2025 ticks, refit cointegration + Markov regimes on a **3-day rolling window**, trade the **next day**, and **flatten at end-of-day** so each day is an independent experiment. No outer WFO — fixed trading-rule params.

In [ ]:
import os, sys, warnings
sys.path.append(os.path.abspath('../scripts'))

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from spread import SPREAD
from screener import SCREENER
from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET

In [ ]:
# ---- Pair + data window ----
NAME_A, NAME_B = 'GBPUSD', 'EURUSD'
PAIR_NAME      = f'{NAME_A}_{NAME_B}'
MONTHS         = ['202501']
DATA_DIR       = '../data/processed'

# ---- Bar aggregation ----
# threshold=500 (vs 1000 in the big notebooks) gives ~2x more bars per day,
# which is what makes a 3-day training window viable for the HMM.
START_HOUR, END_HOUR = 0, 24
BAR_THRESHOLD        = 500

# ---- Engine (per-day rolling fit) ----
TRAIN_DAYS    = 3       # rolling training history
COINT_WINDOW  = 150     # rolling OLS window for beta/alpha
Z_WINDOW      = 50      # rolling z-score lookback
K_REGIMES     = 2       # quiet vs danger
WINSORIZE_STD = 4.0
SCALING       = 10000

# ---- Backtester trading-rule params (fixed; no WFO on a one-month slice) ----
Z_QUIET          = 1.3
Z_VOLATILE       = 2.5
EXIT_Z           = 0.0
DANGER_THRESHOLD = 0.30
FEE_BPS          = 0.5
SLIPPAGE_MODE    = 'half_spread'

# Treat each day as an independent trade: positions reset at EOD.
FLATTEN_EOD = True

In [ ]:
def make_files(name_a, name_b, months):
    a, b = name_a.lower(), name_b.lower()
    return [
        [f'{DATA_DIR}/{a}_dukascopy_ask_{m}.parquet' for m in months],
        [f'{DATA_DIR}/{a}_dukascopy_bid_{m}.parquet' for m in months],
        [f'{DATA_DIR}/{b}_dukascopy_ask_{m}.parquet' for m in months],
        [f'{DATA_DIR}/{b}_dukascopy_bid_{m}.parquet' for m in months],
    ]

builder = SPREAD(
    agg_type='tick',
    threshold=BAR_THRESHOLD,
    active_hours=(START_HOUR, END_HOUR),
)
df = builder.build(make_files(NAME_A, NAME_B, MONTHS))

print(f'Bars  : {len(df):,}')
print(f'Range : {df.index.min()}  ->  {df.index.max()}')
print(f'Days  : {df.index.normalize().unique().shape[0]}')

In [ ]:
screener = SCREENER(df['Asset_A'], df['Asset_B'])
p_val, hl = screener.generate_report(rolling_window=1000, rolling_step=100)

In [ ]:
live, df_params = ENGINE.walk_forward(
    df=df,
    train_days=TRAIN_DAYS,
    coint_window=COINT_WINDOW,
    z_window=Z_WINDOW,
    k_regimes=K_REGIMES,
    winsorize_std=WINSORIZE_STD,
    scaling=SCALING,
    print_freq=5,
)
print(f'\nOOS bars: {len(live):,}  |  Folds: {len(df_params)}')

In [ ]:
bt = BACKTESTER(live)
results = bt.run(
    z_quiet=Z_QUIET,
    z_volatile=Z_VOLATILE,
    exit_z=EXIT_Z,
    danger_threshold=DANGER_THRESHOLD,
    fee_bps=FEE_BPS,
    slippage_mode=SLIPPAGE_MODE,
    flatten_eod=FLATTEN_EOD,
)
print(f'Backtest done. flatten_eod={FLATTEN_EOD}.')

In [ ]:
ts = TEARSHEET(results, df_params=df_params, pdf_prefix=f'{PAIR_NAME}_jan2025')
ts.generate_report()
ts.plot_performance()
ts.plot_positions_and_regimes()
ts.plot_markov_dynamics()
ts.plot_cost_impact()